In [1]:
## imports
import pandas as pd
import numpy as np
# import plotnine
# from plotnine import *
import random

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from datetime import datetime, timedelta

## Load data

In [2]:
## load data on 2020 crimes in DC
df = dc_crim_2020 = pd.read_csv("https://opendata.arcgis.com/datasets/f516e0dd7b614b088ad781b0c4002331_2.csv")

## create report_dt column
df['report_dt'] = pd.to_datetime(df.REPORT_DAT)

## Warm-up Demo

In [3]:
%%time
for i in range(df.shape[0]):
    r = df.iloc[i]
    r.X + r.Y

CPU times: total: 4.12 s
Wall time: 4.18 s


In [4]:
%%time
for i,r in df.iterrows():
    r.X + r.Y

CPU times: total: 2.59 s
Wall time: 2.66 s


In [5]:
%%time
df.apply(lambda r: r.X + r.Y, axis = 1)

CPU times: total: 812 ms
Wall time: 824 ms


0        534560.0
1        535553.0
2        534219.0
3        534863.0
4        533968.0
           ...   
27927    543971.0
27928    538173.0
27929    539531.0
27930    538757.0
27931    532367.0
Length: 27932, dtype: float64

In [6]:
%%time
## Super fast, but only works with built-in numpy functions.
df.X + df.Y

CPU times: total: 0 ns
Wall time: 1.23 ms


0        534560.0
1        535553.0
2        534219.0
3        534863.0
4        533968.0
           ...   
27927    543971.0
27928    538173.0
27929    539531.0
27930    538757.0
27931    532367.0
Length: 27932, dtype: float64

# Practice

In [14]:
## define crimes to look for and crimes to look within
## CCN is Central Complaint Number: https://go.mpdconline.com/GO/GO_401_01.pdf
CCN_examples = ['20165648', '20123250']
C_Tar = C_Target = crimes_lookfor = df[df.CCN.astype(str).isin(CCN_examples)][['CCN', 'WARD', 'OFFENSE', 'report_dt']]
C_Oth = C_Other  = other_crimes = df[~df.CCN.astype(str).isin(CCN_examples)]

## print crimes_lookfor
C_Tar.head()
# other_crimes.head()

,CCN,WARD,OFFENSE,report_dt
7198,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00
12139,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00


**Task**: we have two crimes we want to look for. We want to look in the remaining crime reports for crime reports that are:

- Located in the same ward as the two focal crimes
- Reported at the same time as the focal crime or up to 1000 minutes later (changed from slides which stated 20 mins since crime ids changed since last time so this long bandwidth helps us find matches!)

Solutions compare two ways to solve:

- Using a for loop
- Using a function

## 1. Loop approach

In [8]:
## create empty container to store results 
store_matches = {}

## loop through two example crimes
for i in range(C_Tar.shape[0]): # same as shape
    
    ## extract row
    r = one_row = C_Tar.iloc[i]

    ## first, subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    
    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt +  timedelta(minutes=1200)
    
    ### substep: use that to subset
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()
    
    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime
    
## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

X         Y       CCN              REPORT_DAT  \
20165648 7131   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
         7132   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
         8089   398651.0  136899.0  20166039  2020/11/20 22:07:10+00   
         13609  400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
         13611  399489.0  137478.0  20165986  2020/11/20 22:17:27+00   

                            START_DATE                END_DATE  \
20165648 7131   2020/11/19 23:43:15+00                     NaN   
         7132   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
         8089   2020/11/20 17:30:16+00  2020/11/20 22:08:28+00   
         13609  2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
         13611  2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   

                                                     BLOCK  \
20165648 7131    600 - 669 BLOCK OF PENNSYLVANIA AVENUE SE   
         7132          600 - 699 BLOCK OF ORLEANS PLACE NE   
         8089   300 - 363 BLOCK OF MASSACHUSETTS AVENUE NW   
         13609              800 - 899 BLOCK OF H STREET NE   
         13611          1151 - 1199 BLOCK OF 1ST STREET NE   

                            OFFENSE  METHOD    SHIFT  ...  CENSUS_TRACT  \
20165648 7131           THEFT/OTHER  OTHERS      DAY  ...        6500.0   
         7132          THEFT F/AUTO  OTHERS      DAY  ...       10602.0   
         8089           THEFT/OTHER  OTHERS  EVENING  ...        5900.0   
         13609          THEFT/OTHER  OTHERS      DAY  ...        8402.0   
         13611  MOTOR VEHICLE THEFT  OTHERS  EVENING  ...       10603.0   

               VOTING_PRECINCT           BID    XBLOCK    YBLOCK   LATITUDE  \
20165648 7131      Precinct 89  CAPITOL HILL  400232.0  135255.0  38.885133   
         7132      Precinct 83           NaN  400233.0  137456.0  38.904961   
         8089     Precinct 143      DOWNTOWN  398651.0  136899.0  38.899942   
         13609     Precinct 82           NaN  400489.0  136927.0  38.900195   
         13611    Precinct 144          NOMA  399489.0  137478.0  38.905159   

                LONGITUDE   OBJECTID OCTO_RECORD_ID                 report_dt  
20165648 7131  -76.997326  981157643            NaN 2020-11-20 12:46:32+00:00  
         7132  -76.997314  981157644            NaN 2020-11-20 14:45:06+00:00  
         8089  -77.015552  981164991            NaN 2020-11-20 22:07:10+00:00  
         13609 -76.994363  981209755            NaN 2020-11-20 15:37:59+00:00  
         13611 -77.005891  981209757            NaN 2020-11-20 22:17:27+00:00  

[5 rows x 26 columns]

# 1.5 Iterrow Approach

In [12]:
## create empty container to store results 
store_matches = {}

## loop through two example crimes
for i, r in C_Tar.iterrows(): # same as 

    ## subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    
    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt +  timedelta(minutes=1200)
    
    ### substep: use that to subset
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()
    
    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime
    
## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

X         Y       CCN              REPORT_DAT  \
20123250 870    395618.0  138388.0  20123422  2020/08/29 16:45:57+00   
         9593   396546.0  137533.0  20123507  2020/08/29 22:04:46+00   
         17651  396662.0  138429.0  20401318  2020/08/29 14:29:59+00   
         23220  396523.0  137976.0  20123389  2020/08/29 16:05:18+00   
         23796  398098.0  136808.0  20123419  2020/08/29 17:15:19+00   

                            START_DATE                END_DATE  \
20123250 870    2020/08/26 22:00:29+00  2020/08/27 12:00:51+00   
         9593   2020/08/27 19:01:24+00  2020/08/29 19:00:05+00   
         17651  2020/08/28 20:55:00+00  2020/08/28 21:05:00+00   
         23220  2020/08/28 22:00:23+00  2020/08/29 08:00:27+00   
         23796  2020/08/29 16:05:40+00  2020/08/29 16:08:33+00   

                                                BLOCK              OFFENSE  \
20123250 870    2200 - 2399 BLOCK OF DECATUR PLACE NW         THEFT F/AUTO   
         9593        1700 - 1779 BLOCK OF M STREET NW  MOTOR VEHICLE THEFT   
         17651    1724 - 1799 BLOCK OF 17TH STREET NW          THEFT/OTHER   
         23220       1700 - 1799 BLOCK OF P STREET NW         THEFT F/AUTO   
         23796       700 - 799 BLOCK OF 7TH STREET NW          THEFT/OTHER   

                METHOD    SHIFT  ...  CENSUS_TRACT VOTING_PRECINCT  \
20123250 870    OTHERS      DAY  ...        4100.0     Precinct 13   
         9593   OTHERS  EVENING  ...       10700.0     Precinct 17   
         17651  OTHERS      DAY  ...        5302.0     Precinct 15   
         23220  OTHERS      DAY  ...        5303.0     Precinct 15   
         23796  OTHERS      DAY  ...        5801.0    Precinct 129   

                            BID    XBLOCK    YBLOCK   LATITUDE  LONGITUDE  \
20123250 870                NaN  395618.0  138388.0  38.913346 -77.050526   
         9593   GOLDEN TRIANGLE  396546.0  137533.0  38.905648 -77.039822   
         17651              NaN  396662.0  138429.0  38.913720 -77.038489   
         23220              NaN  396523.0  137976.0  38.909638 -77.040089   
         23796         DOWNTOWN  398098.0  136808.0  38.899121 -77.021926   

                 OBJECTID OCTO_RECORD_ID                 report_dt  
20123250 870    980911088            NaN 2020-08-29 16:45:57+00:00  
         9593   981173732            NaN 2020-08-29 22:04:46+00:00  
         17651  981388845            NaN 2020-08-29 14:29:59+00:00  
         23220  981436904            NaN 2020-08-29 16:05:18+00:00  
         23796  981442283            NaN 2020-08-29 17:15:19+00:00  

[5 rows x 26 columns]

## 2. Function approach

Practice rewriting the above loop as a function

### 2.1 define the function

In [37]:
store_matches_2 = {}

def find_related_crimes(r): # imagine the function taking in one row as its sole variable
    # Your code here
    same_wards = C_Oth[C_Oth.WARD == r.WARD]

    CUTOFF = r.report_dt +  timedelta(minutes=1200)
    
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()

    store_matches_2[str(r.CCN)] = same_wards_sametime

    all_matches = pd.concat(store_matches_2)

    return all_matches

### 2.2 apply it to one of the focal crimes

In [38]:
find_related_crimes(C_Tar.iloc[0])

X         Y       CCN              REPORT_DAT  \
20165648 7131   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
         7132   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
         8089   398651.0  136899.0  20166039  2020/11/20 22:07:10+00   
         13609  400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
         13611  399489.0  137478.0  20165986  2020/11/20 22:17:27+00   
         13650  400042.0  135959.0  20165709  2020/11/20 04:27:36+00   
         13651  399886.0  136784.0  20165932  2020/11/20 18:56:18+00   
         15149  400233.0  137456.0  20165805  2020/11/20 15:06:04+00   

                            START_DATE                END_DATE  \
20165648 7131   2020/11/19 23:43:15+00                     NaN   
         7132   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
         8089   2020/11/20 17:30:16+00  2020/11/20 22:08:28+00   
         13609  2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
         13611  2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   
         13650  2020/11/20 03:02:27+00                     NaN   
         13651  2020/11/20 15:30:02+00  2020/11/20 18:25:35+00   
         15149  2020/11/19 22:30:39+00  2020/11/20 03:00:43+00   

                                                     BLOCK  \
20165648 7131    600 - 669 BLOCK OF PENNSYLVANIA AVENUE SE   
         7132          600 - 699 BLOCK OF ORLEANS PLACE NE   
         8089   300 - 363 BLOCK OF MASSACHUSETTS AVENUE NW   
         13609              800 - 899 BLOCK OF H STREET NE   
         13611          1151 - 1199 BLOCK OF 1ST STREET NE   
         13650            100 - 199 BLOCK OF 5TH STREET NE   
         13651              300 - 399 BLOCK OF G STREET NE   
         15149         600 - 699 BLOCK OF ORLEANS PLACE NE   

                            OFFENSE  METHOD     SHIFT  ...  CENSUS_TRACT  \
20165648 7131           THEFT/OTHER  OTHERS       DAY  ...        6500.0   
         7132          THEFT F/AUTO  OTHERS       DAY  ...       10602.0   
         8089           THEFT/OTHER  OTHERS   EVENING  ...        5900.0   
         13609          THEFT/OTHER  OTHERS       DAY  ...        8402.0   
         13611  MOTOR VEHICLE THEFT  OTHERS   EVENING  ...       10603.0   
         13650  MOTOR VEHICLE THEFT  OTHERS  MIDNIGHT  ...        8200.0   
         13651         THEFT F/AUTO  OTHERS       DAY  ...        8301.0   
         15149         THEFT F/AUTO  OTHERS       DAY  ...       10602.0   

               VOTING_PRECINCT           BID    XBLOCK    YBLOCK   LATITUDE  \
20165648 7131      Precinct 89  CAPITOL HILL  400232.0  135255.0  38.885133   
         7132      Precinct 83           NaN  400233.0  137456.0  38.904961   
         8089     Precinct 143      DOWNTOWN  398651.0  136899.0  38.899942   
         13609     Precinct 82           NaN  400489.0  136927.0  38.900195   
         13611    Precinct 144          NOMA  399489.0  137478.0  38.905159   
         13650     Precinct 89           NaN  400042.0  135959.0  38.891475   
         13651     Precinct 83           NaN  399886.0  136784.0  38.898907   
         15149     Precinct 83           NaN  400233.0  137456.0  38.904961   

                LONGITUDE   OBJECTID OCTO_RECORD_ID                 report_dt  
20165648 7131  -76.997326  981157643            NaN 2020-11-20 12:46:32+00:00  
         7132  -76.997314  981157644            NaN 2020-11-20 14:45:06+00:00  
         8089  -77.015552  981164991            NaN 2020-11-20 22:07:10+00:00  
         13609 -76.994363  981209755            NaN 2020-11-20 15:37:59+00:00  
         13611 -77.005891  981209757            NaN 2020-11-20 22:17:27+00:00  
         13650 -76.999516  981210013            NaN 2020-11-20 04:27:36+00:00  
         13651 -77.001314  981210014            NaN 2020-11-20 18:56:18+00:00  
         15149 -76.997314  981231301            NaN 2020-11-20 15:06:04+00:00  

[8 rows x 26 columns]

### 2.3 Use apply to cover all the other focal crimes

In [39]:
df.apply(lambda r: find_related_crimes(C_Tar.iloc[r]) for r in range(C_Tar.shape[0] - 1))

IndexError: positional indexers are out-of-bounds